# Bag of words and the vector-space model

Every document becomes a vector, a list of numbers. Once texts are numbers, we can compare and classify them at scale.

Bag of words (BoW): which words matters, but not the order.

Term-document matrix:
- Each word is a row
- Each document is a column
- Each cell says something about the presence of the word in that document:
  - raw term frequency: how many times the word appears;
  - binary BoW: 1 if the word appears at least once, 0 if not.

In [ ]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd  # Provides DataFrames for displaying the term-document matrices.
pd.set_option("display.max_columns", None)  # Shows every column instead of hiding some with "...".
from sklearn.feature_extraction.text import CountVectorizer  # Converts text into word-count features.

source = Path("../data/FITZ-TGG-25_paragraphs")  # Points to the folder containing the paragraph text files.
files = sorted(source.glob("*.txt"))  # Finds all .txt files in that folder and sorts their paths.
paragraphs = [path.read_text(encoding="utf-8").strip() for path in files]  # Reads each file into memory and removes whitespace at its edges; files remain unchanged.
names = [path.stem for path in files]  # Gets each filename without its .txt extension to use as a column label.

vec = CountVectorizer()  # Creates a word counter that lowercases text and splits it into tokens.
X = vec.fit_transform(paragraphs)  # Learns the vocabulary and counts each word in each paragraph; rows are paragraphs and columns are words.

tdm = pd.DataFrame(X.T.toarray(), index=vec.get_feature_names_out(), columns=names)  # Transposes the counts so words are rows and paragraph filenames are columns.

print(tdm.shape)  # Prints the matrix dimensions: (number of distinct words, number of paragraph files).

print(tdm) # Displays the term-document matrix with words as rows and paragraph filenames as columns.

# Print the matrix to an external CSV file
tdm.to_csv("term_document_matrix.csv")
print("Term-document matrix saved to term_document_matrix.csv")




In the paragraphs, there are both paragraphs from the published text and paragraphs from Fitzgerald's drafts. I want to find all paragraphs (drafts or published) close to the text of `par01.txt`:

    “Shall we take anything to drink?” called Daisy from an upper window.
    “I’ll get some whiskey,” answered Tom. He went inside.
    Gatsby turned to me rigidly:
    “I can’t say anything in his house, old sport.”
    “She’s got an indiscreet voice,” I remarked. “It’s full of—” I hesitated.
    “Her voice is full of money,” he said suddenly.


In [ ]:
print(tdm.loc[["voice", "money"]])  # Displays counts for these words; errors if any word is absent from the vocabulary.


### Exercise
Look for articles on "Gatsby" and "race" in the [UB Catalogue](https://ubbern.swisscovery.slsp.ch). Try `Gatsby race` and `race Gatsby` and see if the results change.

Probably the same happens if you search your own emails or if you search in a website, for example in Wikipedia.

All these search engines are based on the presence of a word, independently from its meaning and order.

### Pros and cons of BoW
Pros
- Numerical basis for comparison -> analysis at scale becomes possible.
   
Cons
- No semantic ("gossip" =! "rumours", "arm" body = "arm" weapon)
- No order, no syntax: "Daisy shook her head decisively at Miss Baker" = "Miss Baker shook her head decisively at Daisy"
- Vectors are computationally difficult, because they are high dimensional and sparse (many `0`) --> Calculating similarity is cumbersome
- Not every word is equally important

In [ ]:
# The word counts for "rumours" and "gossip" across all paragraphs.
# There is no way to see that they are close in meaning.
print(tdm.loc[["rumours", "gossip"]])  # Displays counts for these words; errors if any word is absent from the vocabulary.

# Displays the word-count vectors for par98.txt and par99.txt.
print(tdm[["par98_fake", "par99_fake"]])  
# Show only the rows where the word count is greater than zero for these paragraphs.
print(tdm.loc[(tdm["par98_fake"] > 0) | (tdm["par99_fake"] > 0), ["par98_fake", "par99_fake"]])



# TF/IDF

If not every word is equally important (limitation of BoW) -> look for words that are **frequent** in a specific document but **rare** in the corpus. For example, stopwords are not significant.

Score
- **Term Frequency**. How often the word appears in a document. Number of occurrence, relative to the document's length.
- **Inverse Document Frequency**. How rare or common the word is. total number of documents ÷ number of documents containing the word. We take the logarithm of this to tame the huge range of values: word frequencies follow Zipf's law (a few words are extremely frequent, most are very rare).

Example: In a corpus of documents about literary studies, the word "autodiegetic" appears 30 times in document X and few times in other documents. The word 'autodiegetic' is thus particularly relevant to categorise document X compared to the others.

Calculate TF/IDF for the word "voice" in our corpus of paragraphs.


In [ ]:
import math  # Provides the natural logarithm used in the IDF calculation.
import spacy  # Imports SpaCy.


# Loads the small English model.
nlp = spacy.load("en_core_web_sm")


# Create an empty list to store the tokens for each paragraph.
tokens_by_paragraph = []
# Process each paragraph one at a time.
for text in paragraphs:
    paragraph_tokens = []
    # Tokenize the current paragraph.
    processed_text = nlp(text)
    # Examine every token in the paragraph.
    for token in processed_text:
        # Keep only alphabetic words longer than one character.
        if token.is_alpha and len(token.text) > 1:
            paragraph_tokens.append(token)
    # Store the tokens from the current paragraph.
    tokens_by_paragraph.append(paragraph_tokens)


# Set the word whose TF-IDF score will be calculated.
word = "voice"
# Lowercase the word to ensure case-insensitive comparison.
word = word.lower()


# Create an empty list to store the word count for each paragraph.
counts = []
# Count how many times the selected word appears in each paragraph.
for tokens in tokens_by_paragraph:
    word_count = 0
    # Examine every token in the current paragraph.
    for token in tokens:
        # Convert the token to lowercase before comparing it with the target word.
        token_text = token.text.lower()
        if token_text == word:
            word_count = word_count + 1
    # Store the count for the current paragraph.
    counts.append(word_count)


# Create an empty list to store the number of retained tokens in each paragraph.
lengths = []
# Count the retained tokens in each paragraph.
for tokens in tokens_by_paragraph:
    paragraph_length = len(tokens)
    lengths.append(paragraph_length)
# Count how many paragraphs contain the selected word.
df = 0
for count in counts:
    if count > 0:
        df = df + 1


# Calculate the inverse document frequency.
# If the word does not appear in any paragraph, set IDF to zero.
if df > 0:
    number_of_paragraphs = len(paragraphs)
    idf = math.log(number_of_paragraphs / df)
else:
    idf = 0


# Calculate and display the TF-IDF score for each paragraph.
for index in range(len(names)):
    name = names[index]
    count = counts[index]
    length = lengths[index]

    # Calculate term frequency.
    # Use zero if the paragraph contains no retained tokens.
    if length > 0:
        tf = count / length
    else:
        tf = 0

    # Calculate the TF-IDF score.
    tf_idf = tf * idf

    # Display the results for the current paragraph.
    print(
        f"{name}: "
        # f"count={count}  "
        # f"length={length}  "
        f"tf={tf:.4f}  "
        f"idf={idf:.3f}  "
        f"tf-idf={tf_idf:.5f}"
    )
    


### Exercise 

Calculate TF/IDF for "Daisy" across paragraphs.

Create a cell here below and copy paste the code from the previous cell. Change "voice" to "Daisy".

Note: the code now will transform to lower case the word inserted, so you write both "Daisy" and "daisy".

In case of errors: print count and length (now commented out, just remove '#').

### Use `scikit-learn`

Instead of calculating TF/IDF manually as done here above, we can use the Python package `scikit-learn`.

Note: `scikit-learn` uses a slightly different IDF to avoid zeros, so its numbers differ a little from the hand-made ones. The ranking logic is the same.


In [ ]:
# Calculate TF/IDF of "voice" in each paragraph
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")   # also drop English stopwords
T = tfidf.fit_transform(paragraphs)
terms = tfidf.get_feature_names_out()

word = "daisy"  # Try with Daisy

for i, name in enumerate(names):
    row = T[i].toarray().ravel()
    if word in terms:
        idx = terms.tolist().index(word)
        tfidf_score = row[idx]
    else:
        tfidf_score = 0
    print(f"{name}: tf-idf={tfidf_score:.5f}")

### Document tagging

Calculate the 3 most characteristic words of each paragraph.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")   # also drop English stopwords
T = tfidf.fit_transform(paragraphs)
terms = tfidf.get_feature_names_out()

for i, name in enumerate(names):
    row = T[i].toarray().ravel()
    top = row.argsort()[::-1][:5]                 # indices of the 3 highest scores
    print(name, "->", ", ".join(terms[top]))



### In practice
For example...
- Ranking system for information retrieval: Search for three words in a corpus = calculate their combined TF/IDF score for those words in every document. 
- Spam filters. Words like "winner", "free", "guarantee" appear often in spam but rarely in normal emails.
- Document tagging.


### Pros and Cons
Pros:
- highly interpretable
- computational efficient

Cons (as BoW):
- No syntax
- No semantic

### Lemmatisation + TF/IDF

In [ ]:
# Calculate TF/IDF for each paragraph on the lemmas, instead of the original tokens.
lemmas_by_paragraph = []
for tokens in tokens_by_paragraph:
    paragraph_lemmas = [token.lemma_ for token in tokens]
    lemmas_by_paragraph.append(paragraph_lemmas)
# Join the lemmas back into strings for TF-IDF vectorization.
paragraphs_lemmas = [" ".join(lemmas) for lemmas in lemmas_by_paragraph]

# Now you can apply TF-IDF vectorization to the lemmatized paragraphs.
tfidf_lemmas = TfidfVectorizer(stop_words="english")
T_lemmas = tfidf_lemmas.fit_transform(paragraphs_lemmas)
terms_lemmas = tfidf_lemmas.get_feature_names_out()

for i, name in enumerate(names):
    row = T_lemmas[i].toarray().ravel()
    top = row.argsort()[::-1][:5]
    print(name, "->", ", ".join(terms_lemmas[top]))

